# Claude で大量のテキストを安く処理する (Message Batches API)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/14_llm_api/claude_batch.ipynb)

NLP クックブック 14-11 ／ LLM の API (Gemini と Claude)

> API キーが必要 (Colab 左の 🔑 シークレットに登録)

数千〜数万件のテキストを LLM で処理したい。急がないので、できるだけ安く済ませたい。

In [ ]:
!pip install -q anthropic

In [ ]:
import time
import anthropic
from anthropic.types.message_create_params import MessageCreateParamsNonStreaming
from anthropic.types.messages.batch_create_params import Request
from google.colab import userdata

client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"))

reviews = {"r001": "料理がとても美味しくて、店員さんも親切でした。", "r002": "1 時間待たされた上に、注文を間違えられた。",
           "r003": "可もなく不可もなく。駅から近いのは助かる。"}
SYSTEM = "レビューの感情を positive / negative / neutral のいずれか 1 語で答えてください。"

batch = client.messages.batches.create(requests=[
    Request(custom_id=rid, params=MessageCreateParamsNonStreaming(
        model="claude-opus-5-5", max_tokens=1024, system=SYSTEM, messages=[{"role": "user", "content": text}]))
    for rid, text in reviews.items()
])
print(batch.id, batch.processing_status)

In [ ]:
while True:
    batch = client.messages.batches.retrieve(batch.id)
    if batch.processing_status == "ended":
        break
    print("処理中:", batch.request_counts.processing, "件")
    time.sleep(60)

results = {}
for r in client.messages.batches.results(batch.id):   # 結果は順不同で返るので、custom_id で対応付ける
    if r.result.type == "succeeded":
        results[r.custom_id] = "".join(b.text for b in r.result.message.content if b.type == "text").strip()
    else:
        results[r.custom_id] = f"({r.result.type})"
for rid, text in reviews.items():
    print(results[rid], "|", text)

- 非同期で処理される代わりに、料金が通常の半額になる。ほとんどのバッチは 1 時間以内に終わる (最大 24 時間)。
- バッチ ID を控えておけば、Colab のセッションが切れても後から結果を取得できる (結果は 29 日間保持される)。
- 共通の長いシステムプロンプトがあるなら、プロンプトキャッシュと併用するとさらに安くなる。